In [1]:

import os
os.getcwd()

'c:\\dev\\accountant_agent_v3\\laboratoire'

In [2]:
import sys
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY") 
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
MODELS="openai/gpt-oss-120b"
MODEL = "openai:gpt-4o-mini"

llm_groq = init_chat_model(model=MODEL, temperature=0)

c:\dev\accountant_agent_v3\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## EVALUATION

In [7]:
#!/usr/bin/env python3
# ============================================================================
# MOROCCAN ACCOUNTING RAG EVALUATION - REALISTIC VERSION
# With simulated model outputs and RAGAS metrics
# ============================================================================

import json
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple
from dataclasses import dataclass
import re
import sys
from difflib import SequenceMatcher
import subprocess

# Install RAGAS if not present
try:
    from ragas import evaluate
    from ragas.metrics.collections import (
        faithfulness,
        answer_relevancy,
        context_recall,
        context_precision,

    )
    HAS_RAGAS = True
except ImportError:
    HAS_RAGAS = False
    print("⚠️  RAGAS not available - using custom metrics only")

# ============================================================================
# LOAD THE EVALUATION DATA
# ============================================================================

with open("data.json") as f:
    evaluation_data = json.load(f)
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def extract_amounts(text: str) -> List[float]:
    """Extract all numeric amounts from text"""
    pattern = r'(\d+(?:[.,]\d+)?)\s*(?:MAD|USD)?'
    matches = re.findall(pattern, text)
    return [float(m.replace(',', '.')) for m in matches]

def calculate_similarity(expected: str, actual: str) -> float:
    """Calculate string similarity ratio"""
    return SequenceMatcher(None, expected.lower(), actual.lower()).ratio()

def extract_account_codes(text: str) -> List[str]:
    """Extract account codes"""
    pattern = r'\b(\d{4,5})\b'
    return re.findall(pattern, text)

def check_debit_credit_match(expected_debits: float, expected_credits: float, 
                             actual_text: str, expected_debits_val: float = None,
                             expected_credits_val: float = None) -> Tuple[bool, float]:
    """Check if debits match credits"""
    actual_amounts = extract_amounts(actual_text)
    
    if not actual_amounts or len(actual_amounts) < 2:
        return False, 0.0
    
    # Sum all amounts (simplified check)
    actual_sum = sum(actual_amounts)
    expected_sum = expected_debits_val + expected_credits_val if expected_debits_val and expected_credits_val else 0
    
    if expected_sum == 0:
        return False, 0.0
    
    match_ratio = min(actual_sum, expected_sum) / max(actual_sum, expected_sum)
    return match_ratio >= 0.99, match_ratio

def evaluate_account_codes(expected: List[str], actual_text: str) -> float:
    """Evaluate account code correctness"""
    extracted = set(extract_account_codes(actual_text))
    expected_set = set(expected)
    
    if not expected_set:
        return 0.5
    
    matches = len(extracted & expected_set)
    return matches / len(expected_set)

def evaluate_tva_handling(expected_text: str, actual_text: str) -> float:
    """Evaluate TVA calculation and handling"""
    expected_has_tva = '34552' in expected_text or '34551' in expected_text or 'TVA' in expected_text
    expected_has_no_tva = 'Pas de TVA' in expected_text or 'taxe spécifique' in expected_text
    
    actual_has_tva = '34552' in actual_text or '34551' in actual_text or 'TVA' in actual_text
    
    if expected_has_no_tva:
        # Should NOT have TVA
        return 1.0 if not actual_has_tva else 0.5
    elif expected_has_tva:
        # Should have TVA
        return 1.0 if actual_has_tva else 0.4
    
    return 0.5

def calculate_bleu_score(expected: str, actual: str) -> float:
    """Simple BLEU-like score (word overlap)"""
    expected_words = set(expected.lower().split())
    actual_words = set(actual.lower().split())
    
    if not expected_words:
        return 0.0
    
    overlap = len(expected_words & actual_words)
    return overlap / len(expected_words)

# ============================================================================
# MAIN EVALUATION
# ============================================================================

def evaluate_rag_system(eval_data: List[Dict]) -> pd.DataFrame:
    """Comprehensive evaluation comparing expected vs actual outputs"""
    results = []
    
    for case in eval_data:
        case_id = case['case_id']
        expected_entry = case['expected_entry']
        model_output = case['model_output']
        context_retrieved = case['context_retrieved']
        expected_accounts = case['expected_accounts']
        expected_debits = case['expected_debits']
        expected_credits = case['expected_credits']
        
        # 1. Account Code Accuracy
        account_accuracy = evaluate_account_codes(expected_accounts, model_output)
        
        # 2. Debit/Credit Balance
        is_balanced, balance_ratio = check_debit_credit_match(
            expected_debits, expected_credits, model_output,
            expected_debits, expected_credits
        )
        
        # 3. TVA Handling
        tva_accuracy = evaluate_tva_handling(expected_entry, model_output)
        
        # 4. Context Relevance (similarity between context and expected entry)
        context_relevance = calculate_similarity(expected_entry, context_retrieved)
        
        # 5. Output Completeness (similarity between expected and actual)
        output_similarity = calculate_similarity(expected_entry, model_output)
        
        # 6. BLEU-like metric (word overlap)
        bleu_score = calculate_bleu_score(expected_entry, model_output)
        
        # Calculate weighted overall score
        overall_score = (
            account_accuracy * 0.25 +
            balance_ratio * 0.25 +
            tva_accuracy * 0.20 +
            context_relevance * 0.10 +
            output_similarity * 0.15 +
            bleu_score * 0.05
        )
        
        results.append({
            'Case': case_id,
            'Query': case['user_input'][:50] + '...',
            'Account Accuracy': account_accuracy,
            'Balance Match': balance_ratio,
            'TVA Handling': tva_accuracy,
            'Context Quality': context_relevance,
            'Output Match': output_similarity,
            'BLEU Score': bleu_score,
            'Overall': overall_score,
            'Status': '✓' if overall_score >= 0.8 else '△' if overall_score >= 0.6 else '✗'
        })
    
    return pd.DataFrame(results)

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    print('\n' + '='*100)
    print('MOROCCAN ACCOUNTING RAG SYSTEM - REALISTIC EVALUATION')
    print('Comparing Model Outputs vs Expected Results')
    print('='*100 + '\n')
    
    print(f'📊 Total Test Cases: {len(evaluation_data)}\n')
    
    # Run evaluation
    results_df = evaluate_rag_system(evaluation_data)
    
    # Display detailed results
    print('DETAILED RESULTS (Model Output vs Expected):')
    print('-'*100)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 50)
    pd.set_option('display.float_format', '{:.3f}'.format)
    print(results_df.to_string(index=False))
    print()
    
    # ========================================================================
    # AGGREGATE METRICS
    # ========================================================================
    
    print('\n' + '='*100)
    print('AGGREGATE METRICS & COMPONENT BREAKDOWN')
    print('='*100 + '\n')
    
    metrics = {
        'Account Code Accuracy': results_df['Account Accuracy'].mean(),
        'Debit/Credit Balance': results_df['Balance Match'].mean(),
        'TVA Handling': results_df['TVA Handling'].mean(),
        'Context Quality': results_df['Context Quality'].mean(),
        'Output Match (Similarity)': results_df['Output Match'].mean(),
        'BLEU Score (Word Overlap)': results_df['BLEU Score'].mean(),
        'Overall RAG Score': results_df['Overall'].mean()
    }
    
    print('Component Performance:\n')
    for metric, value in metrics.items():
        bar_length = int(value * 50)
        bar = '█' * bar_length + '░' * (50 - bar_length)
        status = '✓ Good' if value >= 0.80 else '△ Fair' if value >= 0.60 else '✗ Poor'
        print(f'{metric:<35} {bar} {value:>7.2%}  {status}')
    
    # ========================================================================
    # DIAGNOSTIC ANALYSIS
    # ========================================================================
    
    print('\n' + '='*100)
    print('DIAGNOSTIC ANALYSIS')
    print('='*100 + '\n')
    
    perfect = (results_df['Overall'] >= 0.95).sum()
    excellent = (results_df['Overall'] >= 0.85).sum()
    good = (results_df['Overall'] >= 0.70).sum()
    acceptable = (results_df['Overall'] >= 0.60).sum()
    poor = (results_df['Overall'] < 0.60).sum()
    
    print(f'Perfect (>= 95%):        {perfect:2d}/{len(results_df)} cases')
    print(f'Excellent (85-95%):      {excellent - perfect:2d}/{len(results_df)} cases')
    print(f'Good (70-85%):           {good - excellent:2d}/{len(results_df)} cases')
    print(f'Acceptable (60-70%):     {acceptable - good:2d}/{len(results_df)} cases')
    print(f'Poor (< 60%):            {poor:2d}/{len(results_df)} cases')
    print()
    
    # Problem areas
    weakest = min(metrics.items(), key=lambda x: x[1])
    strongest = max(metrics.items(), key=lambda x: x[1])
    
    print(f'🏆 Strongest: {strongest[0]}: {strongest[1]:.2%}')
    print(f'⚠️  Weakest:   {weakest[0]}: {weakest[1]:.2%}')
    print()
    
    # Worst performing cases
    worst_cases = results_df.nsmallest(3, 'Overall')
    print('Worst Performing Cases:')
    for _, row in worst_cases.iterrows():
        print(f'  Case {int(row["Case"])}: {row["Overall"]:.2%} - {row["Status"]}')
    
    # ========================================================================
    # SUMMARY
    # ========================================================================
    
    print('\n' + '='*100)
    print('SUMMARY')
    print('='*100 + '\n')
    
    overall = metrics['Overall RAG Score']
    
    if overall >= 0.90:
        status = '✓ EXCELLENT - Production ready'
        icon = '🟢'
    elif overall >= 0.80:
        status = '✓ GOOD - Minor issues'
        icon = '🟢'
    elif overall >= 0.70:
        status = '△ ACCEPTABLE - Needs improvements'
        icon = '🟡'
    elif overall >= 0.60:
        status = '✗ POOR - Significant work needed'
        icon = '🔴'
    else:
        status = '✗ CRITICAL - Major redesign'
        icon = '🔴'
    
    print(f'Overall RAG Score: {overall:.2%}')
    print(f'Status: {icon} {status}\n')
    
    # Recommendations
    print('KEY ISSUES TO ADDRESS:')
    print('-'*100)
    
    if metrics['Account Code Accuracy'] < 0.85:
        print('1. Account Code Selection')
        print('   → Improve retrieval of PCGM account codes from documents')
        print('   → Implement fuzzy matching for similar account names\n')
    
    if metrics['Debit/Credit Balance'] < 0.85:
        print('2. Debit/Credit Balance Validation')
        print('   → Add post-processing validation for entry balance')
        print('   → Include balance check in LLM prompt\n')
    
    if metrics['TVA Handling'] < 0.85:
        print('3. TVA Calculation & Rules')
        print('   → Create comprehensive TVA lookup (20%, 14%, 7%, 0%, exempt)')
        print('   → Handle special cases: assurances, carburant, foreign invoices\n')
    
    if metrics['Output Match (Similarity)'] < 0.75:
        print('4. Output Format Consistency')
        print('   → Standardize journal entry formatting in prompts')
        print('   → Add few-shot examples to LLM prompt\n')
    
    # ========================================================================
    # EXPORT RESULTS
    # ========================================================================
    
    print('='*100)
    print('EXPORT RESULTS')
    print('='*100 + '\n')
    
    # CSV export
    os.makedirs('evaluation', exist_ok=True)
    csv_file = 'evaluation/rag_evaluation_realistic_V3.csv'
    results_df.to_csv(csv_file, index=False)
    print(f'✓ Detailed results: {csv_file}')
    
    # Metrics summary
    metrics_df = pd.DataFrame([metrics])
    metrics_file = 'evaluation/rag_metrics_summary_V3.csv'
    metrics_df.to_csv(metrics_file, index=False)
    print(f'✓ Metrics summary: {metrics_file}')
    
    # JSON export
    json_results = {
        'evaluation': 'Moroccan Accounting RAG System_V3',
        'total_cases': len(evaluation_data),
        'overall_score': float(overall),
        'status': status.replace('✓', '').replace('✗', '').replace('△', '').strip(),
        'metrics': {k: float(v) for k, v in metrics.items()},
        'case_breakdown': results_df[['Case', 'Overall', 'Status']].to_dict('records')
    }
    
    json_file = 'evaluation/rag_evaluation_realistic_V3.json'
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(json_results, f, ensure_ascii=False, indent=2)
    print(f'✓ JSON export: {json_file}')
    
    print('\n' + '='*100)
    print('✓ Evaluation Complete!')
    print('='*100 + '\n')

if __name__ == '__main__':
    main()


MOROCCAN ACCOUNTING RAG SYSTEM - REALISTIC EVALUATION
Comparing Model Outputs vs Expected Results

📊 Total Test Cases: 15

DETAILED RESULTS (Model Output vs Expected):
----------------------------------------------------------------------------------------------------
 Case                                                 Query  Account Accuracy  Balance Match  TVA Handling  Context Quality  Output Match  BLEU Score  Overall Status
    1 Facture Maroc Telecom: 300 MAD TTC. Internet Fibre...             0.667          0.006         1.000            0.579         0.865       0.684    0.590      ✗
    2 Achat d'un PC portable HP Victus pour 8500 MAD che...             0.333          0.694         0.400            0.225         0.590       0.421    0.469      ✗
    3 Paiement loyer bureau: 4000 MAD par chèque. Query:...             1.000          0.107         0.500            0.530         0.910       0.727    0.603      △
    4 Facture Facebook Ads: 100 USD pour publicité. Quer...       

## EVALUATION 2 

In [ ]:

#================================================================
# models import
#================================================================
import json
import pandas as pd
from datasets import Dataset
from langchain_core.messages import HumanMessage, ToolMessage
from ragas import evaluate
from ragas.metrics.collections import (
    faithfulness,
    answer_correctness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from src.agentic_workflow.agent import graph  # Adjust import based on your graph object name

#================================================================
# agent setup and evaluation functions
#================================================================

def run_agent_and_capture(user_input: str) -> dict:
    """Invokes agent.py and collects the model response and retrieved contexts."""
    config = {"configurable": {"thread_id": "ragas_eval_session"}}
    initial_state = {"messages": [HumanMessage(content=user_input)]}

    retrieved_contexts = []
    output_messages = []

    # Run query through LangGraph
    final_state = graph.invoke(initial_state, config=config)

    for msg in final_state["messages"]:
        # Capture retrieved contexts from tool node outputs
        if isinstance(msg, ToolMessage):
            retrieved_contexts.append(str(msg.content))
        # Capture final AI response
        elif msg.type == "ai" and msg.content and not msg.tool_calls:
            output_messages.append(msg.content)

    return {
        "model_output": output_messages[-1] if output_messages else "",
        "retrieved_contexts": (
            retrieved_contexts if retrieved_contexts else ["No context retrieved"]
        ),
    }

#================================================================
# main evaluation function
#================================================================
def main():
    # 2. Load Golden Dataset
    with open("data.json", "r", encoding="utf-8") as f:
        golden_dataset = json.load(f)

    user_inputs = []
    expected_outputs = []
    model_outputs = []
    contexts_list = []

    print(f"🚀 Running agent over {len(golden_dataset)} golden test cases...")

    for case in golden_dataset:
        query = case["user_input"]
        ground_truth = case["expected_output"]

        print(f"Executing Case {case['case_id']}...")
        result = run_agent_and_capture(query)

        user_inputs.append(query)
        expected_outputs.append(ground_truth)
        model_outputs.append(result["model_output"])
        contexts_list.append(result["retrieved_contexts"])

    # 3. Construct HuggingFace Dataset format required by RAGAS
    eval_dict = {
    "user_input": user_inputs,
    "ground_truth": expected_outputs,
    "response": [
        res if res and res.strip() else "No response generated" for res in model_outputs
    ],
    "retrieved_contexts": contexts_list,
}
    dataset = Dataset.from_dict(eval_dict)

    # 4. Evaluate using official RAGAS metrics (No custom functions)
    print("\n📊 Computing RAGAS Metrics (Faithfulness, Correctness, Precision, Recall)...")
    results = evaluate(
        dataset=dataset,
        metrics=[
            faithfulness,
            answer_correctness,
            answer_relevancy,
            context_precision,
            context_recall,
        ],
    )
#================================================================
# report generation
#================================================================
    # 5. Export clean report
    df_results = results.to_pandas()
    os.makedirs("evaluation", exist_ok=True)
    
    # 1. Save full CSV
    csv_path = "evaluation/ragas_evaluation_report_V3.csv"
    df_results.to_csv(csv_path, index=False)

    # 2. Extract key metrics columns for summary reports
    metric_cols = [col for col in ['faithfulness', 'answer_correctness', 'answer_relevancy', 'context_precision', 'context_recall'] if col in df_results.columns]

    # 3. Generate Executive Summary Markdown Report (ragas_summary_report.md)
    summary_md_path = "evaluation/ragas_summary_report.md"
    with open(summary_md_path, "w", encoding="utf-8") as f:
        f.write("# RAGAS Evaluation Summary Report\n\n")
        f.write("## Overall Metric Averages\n\n")
        f.write("| Metric | Score |\n")
        f.write("| :--- | :--- |\n")
        for metric in metric_cols:
            avg_score = df_results[metric].mean()
            f.write(f"| **{metric.replace('_', ' ').title()}** | {avg_score:.4f} |\n")
        
        f.write("\n## Detailed Results Table\n\n")
        # Format table for Markdown summary
        summary_cols = ['user_input'] + metric_cols
        summary_df = df_results[summary_cols].copy()
        summary_df['user_input'] = summary_df['user_input'].apply(lambda x: x.replace('\n', ' ')[:80] + ('...' if len(x) > 80 else ''))
        
        f.write(summary_df.to_markdown(index=False))
        f.write("\n")

    # 4. Generate Detailed Query Breakdown Markdown Report (questions_breakdown.md)
    breakdown_md_path = "evaluation/questions_breakdown.md"
    with open(breakdown_md_path, "w", encoding="utf-8") as f:
        f.write("# RAGAS Evaluation - Per Question Detailed Breakdown\n\n")
        
        for idx, row in df_results.iterrows():
            f.write(f"## Question {idx + 1}\n\n")
            f.write(f"**User Query:**\n> {row['user_input']}\n\n")
            
            if 'ground_truth' in row:
                f.write(f"**Expected Output (Ground Truth):**\n```text\n{row['ground_truth']}\n```\n\n")
                
            if 'response' in row:
                f.write(f"**Model Output:**\n```text\n{row['response']}\n```\n\n")
                
            f.write("**Evaluation Scores:**\n\n")
            f.write("| Metric | Score |\n")
            f.write("| :--- | :--- |\n")
            for metric in metric_cols:
                val = row[metric]
                score_str = f"{val:.4f}" if isinstance(val, (int, float)) and not pd.isna(val) else "N/A"
                f.write(f"| {metric.replace('_', ' ').title()} | {score_str} |\n")
            f.write("\n---\n\n")

    print("\n================ RAGAS EVALUATION SUMMARY ================")
    print(results)
    print(f"\n✅ All artifacts generated successfully:")
    print(f"  - CSV Report        : {csv_path}")
    print(f"  - Summary Report    : {summary_md_path}")
    print(f"  - Query Breakdown   : {breakdown_md_path}")


if __name__ == "__main__":
    main()

C:\Users\VICTUS\AppData\Local\Temp\ipykernel_20944\327050954.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\VICTUS\AppData\Local\Temp\ipykernel_20944\327050954.py:6: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import (
C:\Users\VICTUS\AppData\Local\Temp\ipykernel_20944\327050954.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\VICTUS\AppData\Local\Temp\ipykernel

🚀 Running agent over 10 golden test cases...
Executing Case 1...


2026-09-25 14:44:55,154 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-25 14:44:57,091 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-25 14:44:57,126 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-25 14:44:57,219 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-25 14:45:01,250 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Executing Case 2...


2026-09-25 14:45:03,164 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-25 14:45:03,533 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-25 14:45:03,582 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-25 14:45:03,774 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-25 14:45:07,945 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

Executing Case 10...


2026-09-25 14:48:45,253 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-09-25 14:48:45,253 - INFO - Retrying request to /chat/completions in 21.882000 seconds
2026-09-25 14:49:10,386 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-25 14:49:10,803 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-25 14:49:10,835 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-25 14:49:10,874 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-25 14:49:12,739 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-09-25 14:49:12,741 - INFO - Retrying request to /chat/completions in 27.683000 seconds
2026-09-25 14:49:47,820 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



📊 Computing RAGAS Metrics (Faithfulness, Correctness, Precision, Recall)...


Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]2026-09-25 14:49:54,374 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-25 14:50:00,120 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-25 14:50:04,254 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-25 14:50:05,820 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-25 14:50:11,870 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-25 14:50:12,281 - ERROR - Exception raised in Job[1]: TypeError(Cannot use aembed_text() with a synchronous client. Use embed_text() instead.)
2026-09-25 14:50:13,153 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-25 14:50:13,156 - WARNING - LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
202


================ RAGAS EVALUATION SUMMARY ================
{'faithfulness': 0.5241, 'answer_correctness': nan, 'answer_relevancy': nan, 'context_precision': 0.8147, 'context_recall': 0.8333}

Detailed case report saved to 'evaluation/ragas_evaluation_report_V3.csv'


In [ ]:
# ================================================================
# Models & Imports
# ================================================================
# Agent Graph Import
from src.agentic_workflow.agent import graph
import json
import os
import pandas as pd
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, ToolMessage
from langsmith import Client, evaluate
from langsmith.evaluation import EvaluationResult, run_evaluator

# RAGAS Metrics
from ragas import evaluate as ragas_evaluate
from ragas.metrics.collections import (
    faithfulness,
    answer_correctness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from datasets import Dataset


load_dotenv()

client = Client()

# Global counter to track case numbers during LangSmith evaluation runs
case_counter = 0

# ================================================================
# 1. Agent Target Function
# ================================================================

def predict_agent_answer(inputs: dict) -> dict:
    """Invokes agent.py graph and captures model output and context."""
    global case_counter
    case_counter += 1

    user_input = inputs["user_input"]
    query_preview = user_input.replace("\n", " ")[:60] + ("..." if len(user_input) > 60 else "")
    
    # Printed log for each question integrated into the LLM
    print(f"\n🚀 Running Case {case_counter}: {query_preview}")

    config = {
    "configurable": {
        "thread_id": f"langsmith_eval_case_{case_counter}"}}
    initial_state = {"messages": [HumanMessage(content=user_input)]}

    retrieved_contexts = []
    output_messages = []

    final_state = graph.invoke(initial_state, config=config)

    for msg in final_state["messages"]:
        if isinstance(msg, ToolMessage):
            retrieved_contexts.append(str(msg.content))
        elif msg.type == "ai" and msg.content and not msg.tool_calls:
            output_messages.append(msg.content)

    final_response = output_messages[-1] if output_messages else "No response generated"
    contexts = retrieved_contexts if retrieved_contexts else ["No context retrieved"]

    print(f"✅ Completed Case {case_counter}")

    return {
        "response": final_response,
        "retrieved_contexts": contexts
    }


# ================================================================
# 2. Native LangSmith Evaluator Wrapper (Activates Evaluators UI)
# ================================================================

@run_evaluator
def ragas_langsmith_evaluator(run, example):
    """
    Native LangSmith run evaluator that attaches scores directly 
    to each example in the LangSmith UI.
    """
    user_input = example.inputs.get("user_input", "")
    ground_truth = example.outputs.get("ground_truth", "")
    
    # Extract prediction outputs from the target run
    response = run.outputs.get("response", "")
    contexts = run.outputs.get("retrieved_contexts", ["No context retrieved"])

    eval_dict = {
        "user_input": [user_input],
        "ground_truth": [ground_truth],
        "response": [response],
        "retrieved_contexts": [contexts],
    }
    
    ragas_ds = Dataset.from_dict(eval_dict)
    
    # Run RAGAS metrics per example
    eval_results = ragas_evaluate(
        dataset=ragas_ds,
        metrics=[
            faithfulness,
            answer_correctness,
            answer_relevancy,
            context_precision,
            context_recall,
        ],
    )
    
    # Convert RAGAS results to LangSmith EvaluationResults
    scores = eval_results.to_pandas().iloc[0]
    
    results = []
    for metric_name in ['faithfulness', 'answer_correctness', 'answer_relevancy', 'context_precision', 'context_recall']:
        if metric_name in scores and not pd.isna(scores[metric_name]):
            results.append(
                EvaluationResult(
                    key=metric_name,
                    score=float(scores[metric_name])
                )
            )
            
    return results


# ================================================================
# 3. Dataset Setup
# ================================================================

def setup_langsmith_dataset(dataset_name: str, json_file_path: str = "data.json"):
    if client.has_dataset(dataset_name=dataset_name):
        print(f"📌 Found existing LangSmith dataset: '{dataset_name}'")
        return client.read_dataset(dataset_name=dataset_name)

    print(f"🚀 Creating new LangSmith dataset: '{dataset_name}'...")
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Golden Dataset for Moroccan Accounting RAG Evaluation"
    )

    with open(json_file_path, "r", encoding="utf-8") as f:
        golden_data = json.load(f)

    inputs = [{"user_input": item["user_input"]} for item in golden_data]
    outputs = [{"ground_truth": item["expected_output"]} for item in golden_data]

    client.create_examples(inputs=inputs, outputs=outputs, dataset_id=dataset.id)
    return dataset


# ================================================================
# 4. Main Execution
# ================================================================

def main():
    global case_counter
    case_counter = 0  # Reset counter before starting run

    dataset_name = "Moroccan_Accounting_Golden_V3"
    setup_langsmith_dataset(dataset_name, "data.json")

    print("\n🔍 Starting LangGraph Evaluation Stream...")
    
    # Passing evaluators=[ragas_langsmith_evaluator] activates the Evaluators tab
    eval_results = evaluate(
        predict_agent_answer,
        data=dataset_name,
        evaluators=[ragas_langsmith_evaluator],
        experiment_prefix="RAGAS-LangGraph-Run",
        metadata={"version": "1.0.0"}
    )

    # Export local CSV and Markdown reports
    df_results = eval_results.to_pandas()
    os.makedirs("evaluation", exist_ok=True)
    df_results.to_csv("evaluation/ragas_evaluation_report_V3.csv", index=False)

    print("\n================ EVALUATION SUMMARY ================")
    print("✅ Evaluation complete! All test cases processed and traced to LangSmith.")

if __name__ == "__main__":
    main()

# cenario 1 with 10 dataset of bookkepping
### ======== RAGAS EVALUATION SUMMARY ========
#### {'faithfulness': 0.5241, 'answer_correctness': nan, 'answer_relevancy': nan, 'context_precision': 0.8147, 'context_recall': 0.8333}